# Imports de librerias

In [0]:
from pyspark.sql.functions import col, lower, trim

#Lectura de la Tabla Bronce product_category_name_translation

In [0]:
df = spark.table("`catalog_brazilian-e-commerce`.bronze.product_category_name_translation")

In [0]:
df.display()

# Transformaciones

In [0]:

df = (
    df
    
    # Limpieza de strings
    .withColumn("product_category_name", trim(lower(col("product_category_name"))))
    .withColumn("product_category_name_english", trim(lower(col("product_category_name_english"))))

    # Manejo de nulos
    .fillna({
        "product_category_name": "unknown",
        "product_category_name_english": "unknown"
    })

    # Filtrar registros inválidos
    .filter(col("product_category_name").isNotNull())

    # Deduplicación
    .dropDuplicates(["product_category_name"])
)


# Crear la tabla Silver de product_category_name_translation

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("`catalog_brazilian-e-commerce`.silver.product_category_name_translation")

In [0]:
%sql
select * from `catalog_brazilian-e-commerce`.silver.product_category_name_translation